In [ ]:
import polars as pl

In [ ]:
alt_data_path = 'alt_data.csv'

ref_data_path = 'Taplinski_lab_comparisions.tsv'

In [ ]:
alt_data = pl.read_csv(alt_data_path)

In [ ]:
ref_data = pl.read_csv(ref_data_path, separator='\t')

In [ ]:
current_alt_mz = current_alt['mz'][0]

sorted_entry_index = ref_data['mz'].search_sorted(current_alt_mz)

upper_bound = len(ref_data['mz'])

if sorted_entry_index == 0: # This is the case where the current alt mz would be the first item in the (re)sorted list
    index_of_closest_match = 0 # i.e. the first item in the old list is the closest match
elif sorted_entry_index == upper_bound: # len = index of final item + 1 ; this is the case where the current alt mz would be the last item in the (re)sorted list
    index_of_closest_match = upper_bound - 1 # i.e. the last item in the old list is the closest match
else:
    left = abs(ref_data['mz'][sorted_entry_index - 1] - current_alt_mz)
    right = abs(ref_data['mz'][sorted_entry_index] - current_alt_mz)
    if left < right:
        index_of_closest_match = sorted_entry_index - 1
    else:
        index_of_closest_match = sorted_entry_index

In [ ]:
def tabular_sorted_search(
        alt_data: pl.DataFrame,
        alt_data_col: str,
        ref_data: pl.DataFrame,
        ref_data_col: str,
) -> pl.DataFrame:
    """
    Tabular search for a reference data set in an alternative data set.
    
    Parameters
    ----------
    alt_data : pl.DataFrame
        The alternative data set.
    alt_data_col : str
        The column name in the alternative data set.
    ref_data : pl.DataFrame
        The reference data set.
    ref_data_col : str
        The column name in the reference data set.
    
    Returns
    -------
    pl.DataFrame
        The results of the search.
    """

    ref_data = ref_data.sort(pl.col(ref_data_col), multithreaded=True)

    # Do not sort alt_data because we want to preserve the original order

    upper_bound = len(ref_data)

    new_index_list = []

    for current_item in alt_data[alt_data_col]:

        sorted_entry_index = ref_data[ref_data_col].search_sorted(current_item)

        if sorted_entry_index == 0: # This is the case where the current alt mz would be the first item in the (re)sorted list
            index_of_closest_match = 0 # i.e. the first item in the old list is the closest match
        elif sorted_entry_index == upper_bound: # len = index of final item + 1 ; this is the case where the current alt mz would be the last item in the (re)sorted list
            index_of_closest_match = upper_bound - 1 # i.e. the last item in the old list is the closest match
        else:
            left = abs(ref_data['mz'][sorted_entry_index - 1] - current_item)
            right = abs(ref_data['mz'][sorted_entry_index] - current_item)
            if left < right:
                index_of_closest_match = sorted_entry_index - 1
            else:
                index_of_closest_match = sorted_entry_index

        new_index_list.append(index_of_closest_match)

    return ref_data[new_index_list, :]

In [ ]:
test = tabular_sorted_search(alt_data,'mz',ref_data,'mz')

In [ ]:
test = test.rename({'rt': 'rt_ref', 'mz': 'mz_ref'})

In [ ]:
alt_data.hstack(test)

In [ ]:
tabular_sorted_search(alt_data,'rt',ref_data,'rt')

In [ ]:
x = pl.Series("x",[1,2,8,14,34])

x.clip(4,16)

In [ ]:
def tabular_sorted_search_vectorized(
        alt_data: pl.DataFrame,
        alt_data_col: str,
        ref_data: pl.DataFrame,
        ref_data_col: str,
) -> pl.DataFrame:
    """Vectorized tabular search - much faster."""
    
    ref_data = ref_data.sort(pl.col(ref_data_col), multithreaded=True)
    upper_bound = len(ref_data)
    
    # Vectorized search - operates on entire column at once
    sorted_indices = ref_data[ref_data_col].search_sorted(alt_data[alt_data_col])
    
    # Vectorized distance calculations using Polars expressions
    indices = pl.Series("idx", sorted_indices)
    
    # Clamp indices to valid range and get candidates
    left_idx = (indices - 1).clip(0, upper_bound - 1)
    right_idx = indices.clip(0, upper_bound - 1)
    
    # Calculate distances for both candidates
    left_vals = ref_data[ref_data_col][left_idx]
    right_vals = ref_data[ref_data_col][right_idx]
    
    left_dist = (left_vals - alt_data[alt_data_col]).abs()
    right_dist = (right_vals - alt_data[alt_data_col]).abs()
    
    # Choose closest match vectorially
    closest_idx = pl.when(left_dist < right_dist).then(left_idx).otherwise(right_idx)
    
    return ref_data[closest_idx]

In [ ]:
tabular_sorted_search_vectorized(alt_data,'mz',ref_data,'mz')